# Полный pipeline поиска инсайтов

Notebook объединяет выгрузку базового набора данных и последовательный запуск `planner_agent.insight_pipeline` с промежуточными выводами.

Ожидаемая цепочка:

```text
выгрузка данных -> loaded_df -> enriched_df -> clustered_df -> filtered_df -> analyzed_df -> group_summaries
```

Перед запуском должны быть доступны `spark` и, если нужен MCC-справочник, `eng_pg`. API-ключи в notebook не используются.

In [ ]:
"""Полный debug-notebook выгрузки данных и запуска pipeline поиска инсайтов.

Содержит:
- build_week_windows: расчет исторических окон дат.
- safe_to_spark: безопасная конвертация pandas DataFrame в Spark DataFrame.
- normalize_type_accept: нормализация типа подтверждения CSI.
- load_csi_answers: загрузка CSI-ответов.
- transform_survey_data_clean: разворот CSI-вопросов в широкий формат.
- load_hits_extra: загрузка расширенной информации по сработкам.
- coalesce_duplicate_columns: объединение дублей колонок после merge.
- add_business_channel: добавление бизнес-канала.
- load_atm_mcc: загрузка MCC из истории автомаркировки.
- load_mcc_dictionary: загрузка справочника MCC.
- enrich_mcc_names: обогащение названий MCC.
- load_events_for_source: загрузка событий за день по одной source-таблице.
- load_day_events: добавление событий клиента за день.
- build_base_case_dataset: сборка базового датасета для агента.
- show_df: компактный вывод формы, колонок и примеров DataFrame.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyspark.sql.types as T
from IPython.display import display
from pyspark.sql import DataFrame as SparkDataFrame
from pyspark.sql import functions as F

from planner_agent.insight_pipeline import (
    CaseAgentProcessor,
    GroupProblemSummarizer,
    InsightPipeline,
    InsightPipelineConfig,
    InsightPipelineRun,
    NoopDataEnricher,
    StubCommentClusterer,
    StubSignificantGroupSelector,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 300)

## 1. Параметры выгрузки

Все таблицы и основные списки колонок вынесены в конфигурацию. Новые поля из `hits_extra` добавляй в `HITS_EXTRA_COLUMNS`, новые поля из `uko_event` - в `EVENT_EXTRA_COLUMNS` и при необходимости в `EVENT_STRUCT_COLUMNS`.

In [ ]:
@dataclass(frozen=True)
class ExportConfig:
    """Конфигурация выгрузки базового датасета.

    Args:
        date_from: Начальная дата целевого периода в формате YYYY-MM-DD.
        date_to: Конечная дата целевого периода в формате YYYY-MM-DD.
        digest_week: Номера недель дайджеста.
        history_weeks: Количество недель истории для расширенного окна.
        csi_table: Таблица CSI-ответов.
        hits_extra_table: Таблица расширенной информации по сработкам.
        automarking_table: Таблица истории автомаркировки.
        cards_event_table: Таблица card-событий.
        uko_event_table: Таблица UKO-событий.

    Returns:
        Объект с параметрами выгрузки.
    """

    date_from: str = "2026-03-28"
    date_to: str = "2026-04-10"
    digest_week: list[int] = field(default_factory=lambda: [14, 15])
    history_weeks: int = 6
    csi_table: str = "csp_repo_features3.history_csi_clients_answers_v2_129372114_129377108"
    hits_extra_table: str = "cspfs_repo_features3.hits_extra_info_129372427_view"
    automarking_table: str = "csp_repo_features.history_automarking_big_148078_155487"
    cards_event_table: str = "csp_afpc_sss_inc.cards_event"
    uko_event_table: str = "csp_afpc_sss_inc.uko_event"


EXPORT_CONFIG = ExportConfig()

HITS_EXTRA_COLUMNS = [
    "epk_id", "event_id", "event_dt", "event_channel", "sub_channel", "event_type", "age",
    "segment", "age_category", "resolution_first", "resolution_first_dttm",
    "resolution_last", "resolution_last_dttm", "surface", "atm_merchant_name",
    "merchant_info", "source_type_accept", "hits_extra_facts", "client_balance",
    "recipient_info", "scoring_oss", "previous_events_additional_info",
    "posterious_events_additional_info", "previous_events", "posterious_events",
]

CHANNEL_BY_SUB_CHANNEL = {
    "UFS.MOBILEAPI": "ДБО", "ISSUER_ACQUIRER": "Эмиссия", "UFS.BRANCHAPI": "ВСП",
    "ISSUER": "Эмиссия", "UFS.WEBAPI": "ДБО", "ESA.MOBILEAPI": "ДБО",
    "ISSUER_WEBACQUIRER": "Эмиссия", "ATMAPI": "ДБО", "UFS.MBKAPI": "ДБО",
    "UFS.ATMAPI": "ДБО", "UFS.OTHER": "ДБО", "ESA.WEBAPI": "ДБО",
    "ESA.BRANCHAPI": "ВСП",
}

MCC_NAME_OVERRIDES = {
    "6009": "Микрофинансовые организации", "3990": "Экосистема Яндекса",
    "3991": "Экосистема Сбера", "9390": "Госуслуги", "5262": "Маркетплейсы",
    "9406": "Микрофинансовые организации",
}

EVENT_EXTRA_COLUMNS = [
    "main_rule", "rules", "subrules", "list_geo_1km_user_id", "device_source_sdk",
    "user_ip_location_country_code", "user_ip_location_city",
    "payee_communication_term_with_ak_pf", "virus_names", "event_id",
]

EVENT_STRUCT_COLUMNS = [
    "evt_time", "event_type", "sub_type", "event_channel", "event_description",
    "transaction_amount", "atm_mcc", *EVENT_EXTRA_COLUMNS,
]

In [ ]:
def show_df(df: pd.DataFrame, title: str, rows: int = 3) -> None:
    """Показывает промежуточное состояние DataFrame.

    Args:
        df: DataFrame для вывода.
        title: Название этапа.
        rows: Количество строк для `head`.

    Returns:
        None. Функция печатает shape, список колонок и отображает первые строки.
    """

    print(f"\n=== {title} ===")
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    display(df.head(rows))


def build_week_windows(config: ExportConfig) -> tuple[list[list[Any]], str, str, str]:
    """Рассчитывает недельные окна и технические даты для фильтров Spark.

    Args:
        config: Конфигурация выгрузки.

    Returns:
        Кортеж `(weeks_list, history_date_from, date_from_tst, date_to_tst)`.
    """

    date_to_ts = pd.to_datetime(config.date_to)
    current_week_start = (date_to_ts - pd.Timedelta(days=6)).strftime("%Y-%m-%d")
    weeks_list: list[list[Any]] = []
    for num in range(config.history_weeks, 0, -1):
        start_week = (pd.to_datetime(current_week_start) - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        end_week = (date_to_ts - pd.Timedelta(days=7 * num)).strftime("%Y-%m-%d")
        weeks_list.append([start_week, end_week, config.digest_week[-1] - num])
    weeks_list.append([current_week_start, config.date_to, config.digest_week[-1]])
    history_date_from = weeks_list[0][0]
    date_from_tst = pd.to_datetime(history_date_from).strftime("%Y%m%d")
    date_to_tst = pd.to_datetime(config.date_to).strftime("%Y%m%d")
    return weeks_list, history_date_from, date_from_tst, date_to_tst


weeks_list, history_date_from, date_from_tst, date_to_tst = build_week_windows(EXPORT_CONFIG)
print("Целевой период:", EXPORT_CONFIG.date_from, EXPORT_CONFIG.date_to)
print("Историческое окно:", history_date_from, date_to_tst)
display(pd.DataFrame(weeks_list, columns=["start_week", "end_week", "week_num"]))

## 2. Функции выгрузки и обогащения

Эти функции объединяют твой код выборки данных в один переиспользуемый блок.

In [ ]:
def safe_to_spark(pdf: pd.DataFrame) -> SparkDataFrame:
    """Преобразует pandas DataFrame в Spark DataFrame с явной схемой.

    Args:
        pdf: pandas DataFrame для конвертации.

    Returns:
        Spark DataFrame с предсказуемыми типами колонок.
    """

    prepared = pdf.copy()
    object_columns = prepared.select_dtypes(include=["object"]).columns
    prepared[object_columns] = prepared[object_columns].astype(str).replace("nan", None)
    fields: list[T.StructField] = []
    for column_name, dtype in prepared.dtypes.items():
        dtype_text = str(dtype).lower()
        if column_name == "epk_id":
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "int" in dtype_text:
            fields.append(T.StructField(column_name, T.LongType(), True))
        elif "float" in dtype_text:
            fields.append(T.StructField(column_name, T.DoubleType(), True))
        elif "datetime" in dtype_text:
            fields.append(T.StructField(column_name, T.TimestampType(), True))
        else:
            fields.append(T.StructField(column_name, T.StringType(), True))
    return spark.createDataFrame(prepared, schema=T.StructType(fields))


def normalize_type_accept(column: Any) -> Any:
    """Возвращает Spark-выражение для нормализации типа подтверждения.

    Args:
        column: Spark Column `type_accept`.

    Returns:
        Spark Column с нормализованным значением.
    """

    return (
        F.when(column == "ЕРКЦ", F.lit("ЕРКЦ"))
        .when(column == "HINT Эмиссия", F.lit("HINT Карты"))
        .when(column == "HINT ДБО", F.lit("HINT ДБО"))
        .when(column == "Сотрудник ВСП", F.lit("Сотрудник ВСП"))
        .when(column == "ГПМ", F.lit("ГПМ"))
        .when(column == "IVR", F.lit("IVR"))
        .when(column == "ChipPin", F.lit("Chip+Pin"))
        .when(column == "РО/ЗРО", F.lit("РО/ЗРО"))
        .when(column == "Black List", F.lit("Black List (БА)"))
        .when(column == "Сотрудник Бета", F.lit("Сотрудник Бета"))
        .when(column == "БИО", F.lit("BIO"))
        .when(column == "ДБ", F.lit("ДБ"))
        .when(column == "Подтверждение на близкого", F.lit("Подтверждение на близкого"))
        .when(column == "Сотрудник Гамма", F.lit("Сотрудник Гамма"))
        .otherwise(F.lit("Не определено/None"))
    )


def load_csi_answers(config: ExportConfig, history_date_from: str) -> SparkDataFrame:
    """Загружает CSI-ответы клиентов.

    Args:
        config: Конфигурация выгрузки.
        history_date_from: Начало исторического окна.

    Returns:
        Spark DataFrame с CSI-ответами.
    """

    return (
        spark.table(config.csi_table)
        .filter(F.col("answer_date").between(history_date_from, config.date_to))
        .withColumn("type_accept_corrected", normalize_type_accept(F.col("type_accept")))
    )


def transform_survey_data_clean(df: pd.DataFrame, group_columns: list[str] | None = None, question_count: int = 8) -> pd.DataFrame:
    """Разворачивает CSI-вопросы в широкий формат.

    Args:
        df: pandas DataFrame с CSI-ответами.
        group_columns: Колонки, задающие один кейс.
        question_count: Максимальное количество вопросов.

    Returns:
        pandas DataFrame с одной строкой на кейс.
    """

    if group_columns is None:
        group_columns = [column for column in ["epk_id", "event_id", "answer_date"] if column in df.columns]
    if not group_columns:
        raise ValueError("Нужна хотя бы одна колонка группировки для CSI.")
    result_rows: list[dict[str, Any]] = []
    question_columns = {"question_number", "question_text", "answer_text", "comment_text"}
    for _, group in df.groupby(group_columns, dropna=False):
        base_row: dict[str, Any] = {}
        for column in df.columns:
            if column in question_columns:
                continue
            values = group[column].dropna()
            base_row[column] = values.iloc[0] if not values.empty else None
        questions: dict[int, dict[str, Any]] = {}
        for _, row in group.iterrows():
            try:
                q_num = int(str(row.get("question_number")).strip())
            except (ValueError, TypeError):
                continue
            if 1 <= q_num <= question_count:
                questions[q_num] = {"question": row.get("question_text"), "answer": row.get("answer_text"), "comment": row.get("comment_text")}
        for q_num in range(1, question_count + 1):
            question_data = questions.get(q_num, {})
            base_row[f"question_{q_num}"] = question_data.get("question")
            base_row[f"answer_{q_num}"] = question_data.get("answer")
            base_row[f"comment_{q_num}"] = question_data.get("comment")
        result_rows.append(base_row)
    return pd.DataFrame(result_rows)

In [ ]:
def load_hits_extra(config: ExportConfig, event_ids: list[str], date_from_tst: str, date_to_tst: str) -> pd.DataFrame:
    """Загружает расширенную информацию по сработкам.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id.
        date_from_tst: Начальная дата в формате YYYYMMDD.
        date_to_tst: Конечная дата в формате YYYYMMDD.

    Returns:
        pandas DataFrame с полями сработки.
    """

    if not event_ids:
        return pd.DataFrame(columns=HITS_EXTRA_COLUMNS)
    return (
        spark.read.table(config.hits_extra_table)
        .filter(F.col("event_dt").between(date_from_tst, date_to_tst))
        .filter(F.col("event_id").isin(event_ids))
        .select(*HITS_EXTRA_COLUMNS)
        .toPandas()
    )


def coalesce_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Объединяет дублирующиеся колонки после merge.

    Args:
        df: DataFrame после merge.

    Returns:
        DataFrame без дублей имен колонок.
    """

    normalized = df.copy()
    normalized.columns = normalized.columns.str.replace("_x$", "", regex=True).str.replace("_y$", "", regex=True)
    normalized = normalized.replace("", np.nan)
    result: dict[str, pd.Series] = {}
    for column_name in normalized.columns.unique():
        columns = [normalized.iloc[:, idx] for idx, column in enumerate(normalized.columns) if column == column_name]
        merged = columns[0]
        for duplicate_column in columns[1:]:
            merged = merged.combine_first(duplicate_column)
        result[column_name] = merged
    return pd.DataFrame(result)


def add_business_channel(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет бизнес-канал по `sub_channel`.

    Args:
        df: DataFrame с колонкой `sub_channel`.

    Returns:
        DataFrame с колонкой `channel`.
    """

    result = df.copy()
    source = result["sub_channel"] if "sub_channel" in result.columns else pd.Series(index=result.index, dtype="object")
    result["channel"] = source.map(CHANNEL_BY_SUB_CHANNEL).fillna("Не определено")
    return result


def load_atm_mcc(config: ExportConfig, event_ids: list[str], date_from_tst: str, date_to_tst: str) -> pd.DataFrame:
    """Загружает MCC по event_id.

    Args:
        config: Конфигурация выгрузки.
        event_ids: Список event_id.
        date_from_tst: Начальная дата в формате YYYYMMDD.
        date_to_tst: Конечная дата в формате YYYYMMDD.

    Returns:
        DataFrame с колонками `event_id`, `mcc_code`.
    """

    if not event_ids:
        return pd.DataFrame(columns=["event_id", "mcc_code"])
    return (
        spark.table(config.automarking_table)
        .withColumn("event_dt", F.date_format(F.col("event_time"), "yyyyMMdd"))
        .filter(F.col("event_dt").between(date_from_tst, date_to_tst))
        .filter(F.col("event_id").isin(event_ids))
        .selectExpr("event_id", "cast(atm_mcc as string) as mcc_code")
        .toPandas()
    )


def load_mcc_dictionary() -> pd.DataFrame:
    """Загружает справочник MCC.

    Args:
        Нет входных аргументов. Используется внешний объект `eng_pg`.

    Returns:
        DataFrame с колонками `mcc_code`, `mcc_name`, `description`.
    """

    mcc_request = """
    SELECT a.*, b.*
    FROM selfdags.mcc_codes a
    LEFT JOIN dashboards.mcc_dictionary b
    ON CAST(a.mcc_code AS INT) = CAST(b.mcc AS INT)
    """
    mcc_pd = pd.read_sql(mcc_request, eng_pg).drop(columns=["mcc", "activity"], errors="ignore")
    mcc_pd = mcc_pd[["mcc_code", "mcc_name", "description"]].drop_duplicates()
    mcc_pd["mcc_code"] = mcc_pd["mcc_code"].astype(str)
    return mcc_pd


def enrich_mcc_names(df: pd.DataFrame, mcc_dictionary: pd.DataFrame) -> pd.DataFrame:
    """Добавляет описание MCC и бизнес-переопределения названий.

    Args:
        df: DataFrame с колонкой `mcc_code`.
        mcc_dictionary: Справочник MCC.

    Returns:
        DataFrame с `mcc_name` и `description`.
    """

    result = df.copy()
    result["mcc_code"] = result["mcc_code"].astype(str)
    result = result.merge(mcc_dictionary, on="mcc_code", how="left")
    for mcc_code, mcc_name in MCC_NAME_OVERRIDES.items():
        result["mcc_name"] = np.where(result["mcc_code"] == mcc_code, mcc_name, result["mcc_name"])
    return result

In [ ]:
def empty_events_schema() -> T.StructType:
    """Создает схему пустого DataFrame для событий.

    Args:
        Нет входных аргументов.

    Returns:
        Spark StructType для unionByName.
    """

    base_fields = [
        T.StructField("epk_id", T.LongType(), True),
        T.StructField("event_dt", T.StringType(), True),
        T.StructField("event_type", T.StringType(), True),
        T.StructField("sub_type", T.StringType(), True),
        T.StructField("event_channel", T.StringType(), True),
        T.StructField("event_description", T.StringType(), True),
        T.StructField("transaction_amount", T.StringType(), True),
        T.StructField("atm_mcc", T.StringType(), True),
        T.StructField("evt_time", T.TimestampType(), True),
    ]
    return T.StructType(base_fields + [T.StructField(column, T.StringType(), True) for column in EVENT_EXTRA_COLUMNS])


def load_events_for_source(table_name: str, keys: SparkDataFrame, has_extra: bool) -> SparkDataFrame:
    """Загружает события за день по точным парам `epk_id`, `event_dt`.

    Args:
        table_name: Название таблицы событий.
        keys: Spark DataFrame с ключами `epk_id`, `event_dt`.
        has_extra: Есть ли в source-таблице UKO extra-поля.

    Returns:
        Spark DataFrame с унифицированной схемой событий.
    """

    if keys.rdd.isEmpty():
        return spark.createDataFrame([], schema=empty_events_schema())
    source = (
        spark.read.table(table_name)
        .withColumn("epk_id", F.col("epk_id").cast("long"))
        .withColumn("event_dt", F.col("event_dt").cast("string"))
    )
    joined = source.join(keys, on=["epk_id", "event_dt"], how="inner")
    select_list = [
        F.col("epk_id").cast("long").alias("epk_id"),
        F.col("event_dt").cast("string").alias("event_dt"),
        F.col("event_type").cast("string").alias("event_type"),
        F.col("sub_type").cast("string").alias("sub_type"),
        F.col("event_channel").cast("string").alias("event_channel"),
        F.col("event_description").cast("string").alias("event_description"),
        F.col("transaction_amount").cast("string").alias("transaction_amount"),
        F.col("atm_mcc").cast("string").alias("atm_mcc"),
        F.from_unixtime(F.col("event_time").cast("long") / 1000).cast("timestamp").alias("evt_time"),
    ]
    if has_extra:
        select_list.extend([F.col(column).cast("string").alias(column) for column in EVENT_EXTRA_COLUMNS])
    else:
        select_list.extend([F.lit(None).cast("string").alias(column) for column in EVENT_EXTRA_COLUMNS])
    return joined.select(*select_list)


def load_day_events(config: ExportConfig, cases_pdf: pd.DataFrame) -> pd.DataFrame:
    """Добавляет события клиента за день сработки.

    Args:
        config: Конфигурация выгрузки.
        cases_pdf: Базовый DataFrame с кейсами.

    Returns:
        DataFrame с колонкой `events_day`.
    """

    hits = safe_to_spark(cases_pdf)
    base_keys = hits.select(F.col("epk_id").cast("long").alias("epk_id"), F.col("event_dt").cast("string").alias("event_dt"), "channel")
    cards_keys = base_keys.filter(F.col("channel") == "Эмиссия").select("epk_id", "event_dt").distinct()
    uko_keys = base_keys.filter((F.col("channel") != "Эмиссия") | F.col("channel").isNull()).select("epk_id", "event_dt").distinct()
    events_cards = load_events_for_source(config.cards_event_table, cards_keys, has_extra=False)
    events_uko = load_events_for_source(config.uko_event_table, uko_keys, has_extra=True)
    all_events = events_cards.unionByName(events_uko)
    events_col = all_events.groupBy("epk_id", "event_dt").agg(F.sort_array(F.collect_list(F.struct(*EVENT_STRUCT_COLUMNS))).alias("events_day"))
    result = hits.join(events_col, on=["epk_id", "event_dt"], how="left")
    for column in ["event_time", "answer_date", "answer_time", "first_resolution_time", "last_resolution_time"]:
        if column in result.columns:
            result = result.withColumn(column, F.date_format(F.col(column), "yyyy-MM-dd HH:mm:ss"))
    return result.toPandas()


def build_base_case_dataset(config: ExportConfig) -> pd.DataFrame:
    """Собирает базовый dataset для передачи агенту.

    Args:
        config: Конфигурация выгрузки.

    Returns:
        pandas DataFrame с CSI, hits_extra, MCC и событиями дня.
    """

    _, history_date_from_local, date_from_tst_local, date_to_tst_local = build_week_windows(config)
    csi_pdf = load_csi_answers(config, history_date_from_local).toPandas()
    show_df(csi_pdf, "CSI raw", rows=2)
    cases_pdf = transform_survey_data_clean(csi_pdf).drop(columns=["answer_8"], errors="ignore")
    cases_pdf["event_id"] = cases_pdf["event_id"].astype(str)
    show_df(cases_pdf, "CSI wide", rows=2)
    event_ids = cases_pdf["event_id"].dropna().astype(str).unique().tolist()
    hits_extra_pdf = load_hits_extra(config, event_ids, date_from_tst_local, date_to_tst_local)
    hits_extra_pdf["event_id"] = hits_extra_pdf["event_id"].astype(str)
    show_df(hits_extra_pdf, "hits_extra", rows=2)
    cases_pdf = coalesce_duplicate_columns(cases_pdf.merge(hits_extra_pdf, on="event_id", how="left"))
    cases_pdf = add_business_channel(cases_pdf)
    show_df(cases_pdf, "after hits_extra + channel", rows=2)
    atm_mcc_pdf = load_atm_mcc(config, event_ids, date_from_tst_local, date_to_tst_local)
    atm_mcc_pdf["event_id"] = atm_mcc_pdf["event_id"].astype(str)
    cases_pdf = cases_pdf.merge(atm_mcc_pdf, on="event_id", how="left")
    mcc_dictionary = load_mcc_dictionary()
    cases_pdf = enrich_mcc_names(cases_pdf, mcc_dictionary)
    show_df(cases_pdf, "after MCC", rows=2)
    result_pdf = load_day_events(config, cases_pdf)
    show_df(result_pdf, "base_case_dataset with events_day", rows=2)
    return result_pdf

## 3. Запуск выгрузки базового датасета

Эта ячейка обращается к Spark/Postgres-источникам. Если ты уже подготовил `result_pandas`, можно закомментировать вызов `build_base_case_dataset(...)` и оставить свой DataFrame.

In [ ]:
# Основной запуск выгрузки данных.
result_pandas = build_base_case_dataset(EXPORT_CONFIG)
show_df(result_pandas, "result_pandas / loaded source for pipeline", rows=3)

## 4. Настройка компонентов pipeline

Здесь можно подключить реальные компоненты:

- `comment_clusterer` - твоя библиотека кластеризации, которая добавляет колонку группы;
- `group_selector` - LLM binary selector по группам;
- `research_agent` - установленный агент как библиотека.

Если переменные не определены, notebook использует заглушки.

In [ ]:
# Настройка колонок pipeline.
PIPELINE_CONFIG = InsightPipelineConfig(
    text_column="comment_1" if "comment_1" in result_pandas.columns else "comment_text",
    group_column="problem_group",
    case_id_column="case_id" if "case_id" in result_pandas.columns else "event_id",
    event_id_column="event_id",
    min_group_size=3,
    max_groups=10,
    max_cases_per_group=None,  # None = обработать все строки filtered_df
    agent_recursion_limit=60,
)

# Подключи свои компоненты через переменные с такими именами до этой ячейки.
clusterer = globals().get("comment_clusterer", StubCommentClusterer())
selector = globals().get("group_selector", StubSignificantGroupSelector())
agent = globals().get("research_agent", None)

case_processor = CaseAgentProcessor(agent=agent, max_string_length=None)
group_summarizer = GroupProblemSummarizer(summarizer=globals().get("group_summarizer", None))

pipeline = InsightPipeline(
    config=PIPELINE_CONFIG,
    enricher=NoopDataEnricher(),  # result_pandas уже обогащен функциями выше
    clusterer=clusterer,
    group_selector=selector,
    case_processor=case_processor,
    group_summarizer=group_summarizer,
)

print("Pipeline config:")
display(PIPELINE_CONFIG.model_dump())
print("clusterer:", type(clusterer).__name__)
print("selector:", type(selector).__name__)
print("agent:", type(agent).__name__ if agent is not None else None)

## 5. Последовательный запуск pipeline по ячейкам

In [ ]:
# 5.1 loaded_df: вход pipeline.
loaded_df = result_pandas.copy()
show_df(loaded_df, "1. loaded_df", rows=3)

In [ ]:
# 5.2 enriched_df: в этом notebook уже обогащен, поэтому NoopDataEnricher возвращает копию.
enriched_df = pipeline.enricher.enrich(loaded_df)
show_df(enriched_df, "2. enriched_df", rows=3)

In [ ]:
# 5.3 clustered_df: добавляется колонка названия группы.
clustered_df = pipeline._cluster(enriched_df)
show_df(clustered_df, "3. clustered_df", rows=3)
if PIPELINE_CONFIG.group_column in clustered_df.columns:
    display(clustered_df[PIPELINE_CONFIG.group_column].value_counts(dropna=False).head(20).rename("group_size").reset_index())

In [ ]:
# 5.4 group_selection_decisions: бинарная классификация каждой группы.
group_selection_decisions = pipeline._classify_groups(clustered_df)
group_selection_decisions_df = pd.DataFrame([decision.model_dump(mode="json") for decision in group_selection_decisions])
show_df(group_selection_decisions_df, "4. group_selection_decisions", rows=20)

In [ ]:
# 5.5 filtered_df: оставляем только группы, которые selector признал значимыми.
filtered_df = pipeline._filter_selected_groups(clustered_df, group_selection_decisions)
show_df(filtered_df, "5. filtered_df", rows=3)
display(filtered_df[PIPELINE_CONFIG.group_column].value_counts(dropna=False).head(20).rename("filtered_group_size").reset_index())

In [ ]:
# 5.6 analyzed_df: обработка каждой записи агентом и добавление колонок отчета.
# Если agent=None, записи получат статус skipped, а prompt сохранится в case_results.
analyzed_df, case_results = pipeline._process_filtered_cases(filtered_df)
show_df(analyzed_df, "6. analyzed_df", rows=3)
case_results_df = pd.DataFrame([record.model_dump(mode="json") for record in case_results])
show_df(case_results_df, "6.1 case_results_df", rows=3)

In [ ]:
# 5.7 group_summaries: суммаризация проблемы в каждой выбранной группе.
selected_groups = [decision.group_name for decision in group_selection_decisions if decision.is_meaningful]
group_summaries = pipeline.group_summarizer.summarize(
    analyzed_df,
    case_results,
    selected_groups=selected_groups,
    config=PIPELINE_CONFIG,
)
group_summaries_df = pd.DataFrame([summary.model_dump(mode="json") for summary in group_summaries])
show_df(group_summaries_df, "7. group_summaries_df", rows=20)

## 6. Сбор результата в InsightPipelineRun

Эта ячейка собирает все промежуточные результаты в такой же объект, который возвращает `pipeline.run(...)`.

In [ ]:
pipeline_run = InsightPipelineRun(
    loaded_df=loaded_df,
    enriched_df=enriched_df,
    clustered_df=clustered_df,
    filtered_df=filtered_df,
    analyzed_df=analyzed_df,
    group_selection_decisions=group_selection_decisions,
    selected_groups=selected_groups,
    case_results=case_results,
    group_summaries=group_summaries,
    metadata={
        "loaded_rows": len(loaded_df),
        "enriched_rows": len(enriched_df),
        "clustered_rows": len(clustered_df),
        "filtered_rows": len(filtered_df),
        "analyzed_rows": len(analyzed_df),
        "selected_groups_count": len(selected_groups),
        "case_results_count": len(case_results),
    },
)

print("metadata:")
display(pipeline_run.metadata)
print("selected_groups:")
display(pipeline_run.selected_groups)

## 7. Сохранение промежуточных результатов

Раскомментируй нужные строки, если хочешь сохранить результаты локально.

In [ ]:
# output_dir = Path("pipeline_outputs")
# output_dir.mkdir(exist_ok=True)
# loaded_df.to_parquet(output_dir / "01_loaded_df.parquet", index=False)
# enriched_df.to_parquet(output_dir / "02_enriched_df.parquet", index=False)
# clustered_df.to_parquet(output_dir / "03_clustered_df.parquet", index=False)
# filtered_df.to_parquet(output_dir / "04_filtered_df.parquet", index=False)
# analyzed_df.to_parquet(output_dir / "05_analyzed_df.parquet", index=False)
# group_selection_decisions_df.to_csv(output_dir / "group_selection_decisions.csv", index=False, encoding="utf-8-sig")
# case_results_df.to_json(output_dir / "case_results.jsonl", orient="records", lines=True, force_ascii=False)
# group_summaries_df.to_csv(output_dir / "group_summaries.csv", index=False, encoding="utf-8-sig")